In [ ]:
# Core
import pandas as pd
import numpy as np
import os
import cv2
import gc
import re
import copy
import itertools
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from tqdm.notebook import tqdm
from datetime import datetime
import json,itertools
from typing import Optional
from glob import glob
import warnings
from IPython import display as ipd
warnings.filterwarnings("ignore")
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib as mpl
from matplotlib.patches import Rectangle
import seaborn as sns
import random
from joblib import Parallel, delayed
import os, shutil
import datetime 
import holidays
import dateutil.easter as easter

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet,SGDRegressor
from sklearn.model_selection import StratifiedGroupKFold

from xgboost import XGBRegressor
import xgboost as xgb
import lightgbm as lgb
from xgboost import plot_importance

from xgboost import plot_importance
from matplotlib import pyplot
import missingno as msno
import plotly.express as px

import optuna

# Keras
import tensorflow as tf

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from lightgbm import LGBMRegressor

# Keras
from tensorflow import keras
import tensorflow as tf
import keras
from keras import backend as K
from keras.models import Model
from tensorflow.keras.layers import LSTM, Flatten, TimeDistributed, Conv1D, Input, Dense, Multiply, Add, Activation, GRU, BatchNormalization
from keras.layers.convolutional import Conv2D, Conv2DTranspose
from keras.layers.pooling import MaxPooling2D
from keras.losses import binary_crossentropy
from keras.callbacks import Callback, ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.models import load_model, save_model, Sequential
from tensorflow.data import Dataset
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras import optimizers

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Conv1D, Flatten,MaxPooling1D,BatchNormalization, Lambda, AveragePooling1D, Dropout, Input
from keras.callbacks import EarlyStopping, ModelCheckpoint,ReduceLROnPlateau
import tensorflow.keras as keras

## Import the Dataset

In [ ]:
train_df = pd.read_csv('../input/tabular-playground-series-sep-2022/train.csv', parse_dates=['date'])
original_train_df = train_df.copy()
test_df = pd.read_csv('../input/tabular-playground-series-sep-2022/test.csv', parse_dates=['date'])
train_df.head()

In [ ]:
df_gdp = pd.read_csv("../input/tpssep22-gdp-data-20172021/TPSSEP22_GDP_data_2017_to_2021.csv")
df_gdp_all = df_gdp.groupby('year')['GDP'].sum().reset_index()

In [ ]:
for df in [train_df, test_df]:
    df['product'] = df['product'].str.replace(' ', '_')
    df['product'] = df['product'].str.replace(':', '_')

In [ ]:
test_df.head()

## Explore the Dataset

In [ ]:
train_df.isnull().sum()

In [ ]:
test_df.isnull().sum()

In [ ]:
print('\n  country in the train dataset\n')
print(train_df['country'].value_counts())

print('\n  country in the test dataset\n')
print(test_df['country'].value_counts())

print('\n store in the train dataset\n')
print(train_df['store'].value_counts())

print('\n store in the test dataset\n')
print(test_df['store'].value_counts())

print('\n product in the train dataset\n')
print(train_df['product'].value_counts())

print('\n product in the test dataset\n')
print(test_df['product'].value_counts())

In [ ]:
print("train min date:", train_df['date'].min())
print("train max date:", train_df['date'].max())
print("test min date:", test_df['date'].min())
print("test max date:", test_df['date'].max())

## Data Visualization

In [ ]:
plt.figure(figsize=(15,6))
train_df.groupby(['country','store','product'])['num_sold'].mean().unstack().plot(kind='bar',stacked=True)
plt.show()

In [ ]:
plt.figure(figsize=(15,6))
train_gp = train_df.groupby('date').sum().reset_index()
plt.plot(train_gp['date'], train_gp['num_sold'])
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))
for country in train_df['country'].unique():
    filt_train = train_df[train_df['country'] == country]

    train_gp = filt_train.groupby('date').sum().reset_index()
    plt.plot(train_gp['date'], train_gp['num_sold'], label=country)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))
for store in train_df['store'].unique():
    filt_train = train_df[train_df['store'] == store]

    train_gp = filt_train.groupby('date').sum().reset_index()
    plt.plot(train_gp['date'], train_gp['num_sold'], label=store)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))
for product in train_df['product'].unique():
    filt_train = train_df[train_df['product'] == product]

    train_gp = filt_train.groupby('date').sum().reset_index()
    plt.plot(train_gp['date'], train_gp['num_sold'], label=product)
plt.legend()
plt.show()

In [ ]:
weekly_df = train_df.groupby(["country","store", "product", pd.Grouper(key="date", freq="W")])["num_sold"].sum().rename("num_sold").reset_index()
monthly_df = train_df.groupby(["country","store", "product", pd.Grouper(key="date", freq="MS")])["num_sold"].sum().rename("num_sold").reset_index()

In [ ]:
plt.figure(figsize=(15,6))
for product in train_df['product'].unique():
    filt_train = weekly_df[weekly_df['product'] == product]

    train_gp = filt_train.groupby('date').sum().reset_index()
    plt.plot(train_gp['date'], train_gp['num_sold'], label=product)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))
for product in train_df['product'].unique():
    filt_train = monthly_df[monthly_df['product'] == product]

    train_gp = filt_train.groupby('date').sum().reset_index()
    plt.plot(train_gp['date'], train_gp['num_sold'], label=product)
plt.legend()
plt.show()

In [ ]:
product_store_weights = monthly_df.groupby(["product","store"])["num_sold"].sum() / monthly_df.groupby(["product"])["num_sold"].sum()
product_store_weights

In [ ]:
store_weights = train_df.groupby("store")["num_sold"].sum()/train_df["num_sold"].sum()
store_weights

In [ ]:
new_monthly_df = monthly_df.loc[monthly_df["date"] < "2020-01-01"]
product_country_weights = new_monthly_df.groupby(["product","country"])["num_sold"].sum() / new_monthly_df.groupby(["product"])["num_sold"].sum()
product_country_weights

In [ ]:
(product_country_weights.reset_index().groupby("country")["num_sold"].mean().loc["Belgium"] / product_country_weights.reset_index().groupby("country")["num_sold"].mean())

In [ ]:
product_df = train_df.groupby(["date","product"])["num_sold"].sum().reset_index()
product_df.head()

In [ ]:
product_ratio_df = product_df.pivot(index="date", columns="product", values="num_sold")
product_ratio_df = product_ratio_df.apply(lambda x: x/x.sum(),axis=1)
product_ratio_df = product_ratio_df.stack().rename("ratios").reset_index()
product_ratio_df.head(4)

In [ ]:
# product ratio 관련 예측 코드 추가해보기

In [ ]:
product_ratio_df.tail()

In [ ]:
f,ax = plt.subplots(figsize=(20,10))
sns.lineplot(data = product_ratio_df, x="date", y="ratios", hue="product");

## Train and Aggregated Dataset

In [ ]:
train_df = train_df.groupby(["date"])["num_sold"].sum().reset_index()

In [ ]:
weekly_df = train_df.groupby([pd.Grouper(key="date", freq="W")])["num_sold"].sum().rename("num_sold").reset_index()
monthly_df = train_df.groupby([pd.Grouper(key="date", freq="MS")])["num_sold"].sum().rename("num_sold").reset_index()

In [ ]:
train_nocovid_df = train_df.loc[~((train_df["date"] >= "2020-03-01") & (train_df["date"] < "2020-06-01"))]
f,ax = plt.subplots(figsize=(20,10))
sns.lineplot(data = train_nocovid_df, x="date", y="num_sold");

In [ ]:
train_df = train_nocovid_df

#get the dates to forecast for
test_all_df = test_df.groupby(["date"])["row_id"].first().reset_index().drop(columns="row_id")
#keep dates for later
test_all_df_dates = test_all_df[["date"]]

In [ ]:
# https://www.kaggle.com/competitions/optiver-realized-volatility-prediction/discussion/278588

In [ ]:
def feature_engineer(df):
    new_df = df.copy()
    new_df["month"] = df["date"].dt.month
    new_df["month_sin"] = np.sin(new_df['month'] * (2 * np.pi / 12))
    new_df["month_cos"] = np.cos(new_df['month'] * (2 * np.pi / 12))
    
    new_df["day"] = df["date"].dt.day
    new_df["day_sin"] = np.sin(new_df['day'] * (2 * np.pi / 12))
    #new_df["day_cos"] = np.cos(new_df['day'] * (2 * np.pi / 12))
    
    new_df["day_of_week"] = df["date"].dt.dayofweek
    new_df["day_of_week"] = new_df["day_of_week"].apply(lambda x: 0 if x<=3 else(1 if x==4 else (2 if x==5 else (3))))
    
    new_df['friday'] = new_df.date.dt.weekday.eq(4).astype(np.uint8)
    new_df['saturday'] = new_df.date.dt.weekday.eq(5).astype(np.uint8)
    new_df['sunday'] = new_df.date.dt.weekday.eq(6).astype(np.uint8)
    new_df['sat+sun'] = new_df['saturday']+new_df['sunday']
    new_df['fri+sun'] = new_df['friday']+new_df['sunday']
    
    new_df["day_of_year"] = df["date"].dt.dayofyear

    new_df["day_of_year"] = new_df.apply(lambda x: x["day_of_year"]-1 if (x["date"] > pd.Timestamp("2020-02-29") and x["date"] < pd.Timestamp("2021-01-01"))  else x["day_of_year"], axis=1)
    
    important_dates = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,16,17,22,23,24,105, 123,124, 125, 126, 127, 140, 141,142, 167, 168, 169, 170, 171, 173, 174, 175, 176, 177, 178, 179,
                  180, 181, 203, 230, 231, 232, 233, 234, 282, 289, 290, 307, 308, 309, 310, 311, 312, 313, 317, 318, 319, 320, 360, 361, 362, 363, 364, 365]
    

    new_df["important_dates"] = new_df["day_of_year"].apply(lambda x: x if x in important_dates else 0)
    #new_df["important_dates"] = new_df["day_of_year"].apply(lambda x: x if x in [1,2,3,4,5,6,7,8,125,126,360,361,362,363,364,365] else 0)
    
    
    new_df["year"] = df["date"].dt.year
    #new_df['is_pandemic_year'] = new_df['year'].astype(int) >= 2020
    '''
    easter_date = new_df.date.apply(lambda date: pd.Timestamp(easter.easter(date.year)))
    new_df = pd.concat([new_df,
                        pd.DataFrame({f"easter{d}": 
                                      (df.date - easter_date == np.timedelta64(d, "D"))
                                      for d in list(range(-2, 11)) + list(range(40, 48)) + list(range(51, 58))})],
                       axis=1)
    '''
    
    easter_date = new_df.date.apply(lambda date: pd.Timestamp(easter.easter(date.year)))
    for day in list(range(-5, 5)) + list(range(40, 48)):
        new_df[f'easter_{day}'] = (new_df.date - easter_date).dt.days.eq(day)
        
    for col in new_df.columns :
        if 'easter' in col :
            new_df = pd.get_dummies(new_df, columns = [col], drop_first=True)
            
    for day in range(24, 32):
        new_df[f'Dec_{day}'] = new_df.date.dt.day.eq(day) & new_df.date.dt.month.eq(12)
        
        
    new_df = new_df.drop(columns=["date","month","day", "day_of_year"])
    
   
    
    new_df = pd.get_dummies(new_df, columns = ["important_dates","day_of_week"], drop_first=True)
    
    return new_df

In [ ]:
def get_holidays(df):
    years_list = [2017, 2018, 2019, 2020, 2021]

    holiday_BE = holidays.CountryHoliday('BE', years = years_list)
    holiday_FR = holidays.CountryHoliday('FR', years = years_list)
    holiday_DE = holidays.CountryHoliday('DE', years = years_list)
    holiday_IT = holidays.CountryHoliday('IT', years = years_list)
    holiday_PL = holidays.CountryHoliday('PL', years = years_list)
    holiday_ES = holidays.CountryHoliday('ES', years = years_list)

    holiday_dict = holiday_BE.copy()
    holiday_dict.update(holiday_FR)
    holiday_dict.update(holiday_DE)
    holiday_dict.update(holiday_IT)
    holiday_dict.update(holiday_PL)
    holiday_dict.update(holiday_ES)

    df['holiday_name'] = df['date'].map(holiday_dict)
    df['is_holiday'] = np.where(df['holiday_name'].notnull(), 1, 0)
    df['holiday_name'] = df['holiday_name'].fillna('Not Holiday')
    
    return df

In [ ]:
def encode_holiday_names(df, enc, subset='train'):
    if subset=='train':
        df['holiday_name'] = enc.fit_transform(df['holiday_name'].values.reshape(-1,1))
    else:
        df['holiday_name'] = enc.transform(df['holiday_name'].values.reshape(-1,1))
        not_hol_val = oe.transform([['Not Holiday']])[0,0]
        df.loc[df['holiday_name']==-1, 'holiday_name'] = not_hol_val
    return df

In [ ]:
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

In [ ]:
train_all_df = get_holidays(train_df)
test_all_df = get_holidays(test_all_df)

In [ ]:
train_all_df = feature_engineer(train_all_df)
test_all_df = feature_engineer(test_all_df)

In [ ]:
train_all_df = encode_holiday_names(train_all_df, oe)
test_all_df = encode_holiday_names(test_all_df, oe)

In [ ]:
# Add GDP Feature
#train_all_df = train_all_df.merge(df_gdp_all, left_on=['year'], right_on=['year'], how='left')
#test_all_df = test_all_df.merge(df_gdp_all, left_on=['year'], right_on=['year'], how='left')

In [ ]:
#train_all_df.loc[train_all_df[(train_all_df['is_pandemic_year'] == False)]['is_pandemic_year'].index, 'is_pandemic_year'] = 0
#train_all_df.loc[train_all_df[(train_all_df['is_pandemic_year'] == True)]['is_pandemic_year'].index, 'is_pandemic_year'] = 1
#test_all_df.loc[test_all_df[(test_all_df['is_pandemic_year'] == True)]['is_pandemic_year'].index, 'is_pandemic_year'] = 1
#test_all_df.loc[test_all_df[(test_all_df['is_pandemic_year'] == True)]['is_pandemic_year'].index, 'is_pandemic_year'] = 1
#train_all_df['is_pandemic_year'] = train_all_df['is_pandemic_year'].astype('uint')
#test_all_df['is_pandemic_year'] = test_all_df['is_pandemic_year'].astype('uint')

In [ ]:
display(train_all_df.head(2))
display(test_all_df.head(2))

In [ ]:
y = train_all_df["num_sold"]
X = train_all_df.drop(columns="num_sold")
X_test = test_all_df

In [ ]:
def smape(y_true, y_pred):
    smape = abs(y_true - y_pred) / (abs(y_true) + abs(y_pred))
    smape = smape.mean() * 200
    return smape

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=.33)

In [ ]:
linear_model = LinearRegression()
linear_model.fit(x_train, y_train)
linear_preds = linear_model.predict(x_test)

ridge_model = Ridge(tol=1e-2, max_iter=1000000, random_state=42)
ridge_model.fit(x_train, y_train)
ridge_preds = ridge_model.predict(x_test)

lasso_model = Lasso(tol=1e-2, max_iter=1000000, random_state=42)
lasso_model.fit(x_train, y_train)
lasso_preds = lasso_model.predict(x_test)

electic_model = ElasticNet(alpha=0.01, l1_ratio=0.01, random_state=42)
electic_model.fit(x_train, y_train)
electic_preds = electic_model.predict(x_test)

print('lr_pre : ',smape(y_test,linear_preds))
print('ridge_pre : ',smape(y_test,ridge_preds))
print('lasso_pre : ',smape(y_test,lasso_preds))
print('electic_pre : ',smape(y_test,electic_preds))

In [ ]:
'''
ridge_params = {'alpha' :  0.0001 * np.arange(1, 1000)}
grid = GridSearchCV(estimator = ridge_model, param_grid = ridge_params, scoring = 'neg_mean_absolute_error')

grid.fit(X, y)
best_param = grid.best_params_
print(best_param)
'''

In [ ]:
ridge_model2 = Ridge(alpha=0.0999, tol=1e-2, max_iter=1000000, random_state=42)
ridge_model2.fit(x_train, y_train)
ridge_preds2 = ridge_model2.predict(x_test)
print('ridge_pre2 : ',smape(y_test,ridge_preds2))

In [ ]:
'''
lasso_params = {'alpha' :  0.0001 * np.arange(1, 1000) }
grid = GridSearchCV(estimator = lasso_model, param_grid = lasso_params, scoring = 'neg_mean_absolute_error')

grid.fit(X, y)
best_param = grid.best_params_
print(best_param)
'''

In [ ]:
lasso_model2 = Lasso(alpha=0.0099, tol=1e-2, max_iter=1000000, random_state=42)
lasso_model2.fit(x_train, y_train)
lasso_preds2 = lasso_model2.predict(x_test)
print('ridge_pre2 : ',smape(y_test,lasso_preds2))

In [ ]:
'''
electic_param = {
     'alpha': 0.0001 * np.arange(1, 100),
     'l1_ratio': 0.001 * np.arange(1, 10)}

grid = GridSearchCV(estimator = electic_model, param_grid = electic_param, scoring = 'neg_mean_absolute_error')
    
grid.fit(X, y)
best_param = grid.best_params_
print(best_param)
'''

In [ ]:
elastic_model2 = ElasticNet(alpha=0.0002, l1_ratio=0.001, random_state=42)
elastic_model2.fit(x_train, y_train)
elastic_preds2 = elastic_model2.predict(x_test)
print('elastic_pre2 : ',smape(y_test,elastic_preds2))

### Linear Regression(Kfold)

In [ ]:
linear_preds = np.zeros(X_test.shape[0])
avg_smape = 0
n=0

kf = GroupKFold(n_splits=4)

for trn_idx, test_idx in kf.split(X, groups=X.year) :
    x_train, x_valid = X.iloc[trn_idx], X.iloc[test_idx]
    y_train, y_valid = y.iloc[trn_idx], y.iloc[test_idx]
    
    linear_model = LinearRegression()
    #linear_model = make_pipeline(StandardScaler(), linear_model)
    linear_model.fit(x_train, y_train)

    y_pred = linear_model.predict(x_valid)
    avg_smape += smape(y_pred, y_valid)

    n = n + 1

    test_pred = linear_model.predict(X_test)
    pred = pd.Series(test_pred) 

    linear_preds += pred / kf.n_splits  
    
print(f"smape: {avg_smape/kf.n_splits}")

### Ridge(Kfold)

In [ ]:
ridge_preds = np.zeros(X_test.shape[0])
avg_smape = 0
n=0

kf = GroupKFold(n_splits=4)

for trn_idx, test_idx in kf.split(X, groups=X.year) :
    x_train, x_valid = X.iloc[trn_idx], X.iloc[test_idx]
    y_train, y_valid = y.iloc[trn_idx], y.iloc[test_idx]
    
    ridge_model = Ridge(alpha=0.0999, tol=1e-2, max_iter=1000000, random_state=0)
    #ridge_model = make_pipeline(StandardScaler(), ridge_model)
    ridge_model.fit(x_train, y_train)

    y_pred = ridge_model.predict(x_valid)
    avg_smape += smape(y_pred, y_valid)

    n = n + 1

    test_pred = ridge_model.predict(X_test)
    pred = pd.Series(test_pred) 

    ridge_preds += pred / kf.n_splits  
    
print(f"smape: {avg_smape/kf.n_splits}")

### Lasso(Kfold)

In [ ]:
lasso_preds = np.zeros(X_test.shape[0])
avg_smape = 0
n=0

kf = GroupKFold(n_splits=4)

for trn_idx, test_idx in kf.split(X, groups=X.year) :
    x_train, x_valid = X.iloc[trn_idx], X.iloc[test_idx]
    y_train, y_valid = y.iloc[trn_idx], y.iloc[test_idx]
    
    lasso_model = Lasso(alpha=0.0999, tol=1e-2, max_iter=1000000, random_state=0)
    #lasso_model = make_pipeline(StandardScaler(), lasso_model)
    lasso_model.fit(x_train, y_train)

    y_pred = lasso_model.predict(x_valid)
    avg_smape += smape(y_pred, y_valid)

    n = n + 1

    test_pred = lasso_model.predict(X_test)
    pred = pd.Series(test_pred) 

    lasso_preds += pred / kf.n_splits  
    
print(f"smape: {avg_smape/kf.n_splits}")

### ElasticNet(Kfold)

In [ ]:
elastic_preds = np.zeros(X_test.shape[0])
avg_smape = 0
n=0

kf = GroupKFold(n_splits=4)

for trn_idx, test_idx in kf.split(X, groups=X.year) :
    x_train, x_valid = X.iloc[trn_idx], X.iloc[test_idx]
    y_train, y_valid = y.iloc[trn_idx], y.iloc[test_idx]
    
    elastic_model = ElasticNet(alpha=0.0002, l1_ratio=0.001, random_state=42)
    #elastic_model = make_pipeline(StandardScaler(), elastic_model)
    elastic_model.fit(x_train, y_train)

    y_pred = elastic_model.predict(x_valid)
    avg_smape += smape(y_pred, y_valid)

    n = n + 1

    test_pred = elastic_model.predict(X_test)
    pred = pd.Series(test_pred) 

    elastic_preds += pred / kf.n_splits  
    
print(f"smape: {avg_smape/kf.n_splits}")

In [ ]:
def lgbm_objective(trial,data=X,target=y):
    
    train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.2,random_state=42)

        
    param = {'metric': 'mape', 
        'random_state': 48,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.3,0.4,0.5,0.6,0.7,0.8,0.9, 1.0]),
        'subsample': trial.suggest_categorical('subsample', [0.4,0.5,0.6,0.7,0.8,1.0]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.006,0.008,0.01,0.014,0.017,0.02, 0.1, 0.04]),
        'max_depth': trial.suggest_categorical('max_depth', [10,20,100]),
        'num_leaves' : trial.suggest_int('num_leaves', 1, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 300),
        'cat_smooth' : trial.suggest_int('min_data_per_groups', 1, 100),

    }
    model = lgb.LGBMRegressor(**param)  
    
    model.fit(train_x,train_y,eval_set=[(test_x,test_y)],early_stopping_rounds=100,verbose=False)
    
    preds = model.predict(test_x)
    
    metric = smape(test_y, preds)
    
    return metric

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(lgbm_objective, n_trials=30)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

Best_trial=study.best_params   

In [ ]:
lgb_preds = np.zeros(X_test.shape[0])
avg_smape = 0
n=0

kf = GroupKFold(n_splits=4)

Best_trial= {'n_estimators': 391, 'reg_alpha': 0.0010356397081781289, 'reg_lambda': 0.00104105557242766, 'colsample_bytree': 0.9, 'subsample': 1.0, 'learning_rate': 0.04, 'max_depth': 20, 'num_leaves': 996, 'min_child_samples': 2, 'min_data_per_groups': 53}
Best_trial['random_state'] = 42
Best_trial['metric'] = 'mape'

for trn_idx, test_idx in kf.split(X, groups=X.year) :
    x_train, x_valid = X.iloc[trn_idx], X.iloc[test_idx]
    y_train, y_valid = y.iloc[trn_idx], y.iloc[test_idx]
    
    lgb_reg = LGBMRegressor(**Best_trial)

    lgb_reg.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], early_stopping_rounds=100, verbose=False)
    y_pred = lgb_reg.predict(x_valid)

    avg_smape += smape(y_pred, y_valid)

    n = n + 1

    test_pred = lgb_reg.predict(X_test)
    pred = pd.Series(test_pred) 

    lgb_preds += pred / kf.n_splits  
    
print(f"smape: {avg_smape/kf.n_splits}")

In [ ]:
test_all_df_dates["num_sold"] = ridge_preds * 0.4 + lasso_preds * 0.4 + linear_preds * 0.2 + elastic_preds * 0 
#test_all_df_dates["num_sold"] = ridge_preds * 0.5 + lasso_preds * 0.5 + linear_preds * 0 + elastic_preds * 0 


In [ ]:
models = [linear_model,ridge_model, lasso_model, elastic_model]

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

for i in range(len(models)):
    scores = cross_val_score(models[i],X,y,scoring=make_scorer(smape,greater_is_better=False))
    print (models[i],scores, np.mean(scores))

In [ ]:
def get_top_bottom_coef(model, n=8):
    coef = pd.Series(model.coef_,index=X.columns)
    coef_high = coef.sort_values(ascending=False).head(n)
    coef_low = coef.sort_values(ascending=False).tail(n)
    return coef_high, coef_low

def visualize_coefficient(models):
    fig,axs= plt.subplots(figsize=(24,10),nrows=1,ncols=4)
    fig.tight_layout()
    for i_num, model in enumerate(models):
        coef_high,coef_low = get_top_bottom_coef(model)
        coef_concat = pd.concat([coef_high,coef_low])
        axs[i_num].set_title(model.__class__.__name__+' Coefficients',size=25)
        axs[i_num].tick_params(axis='y',direction="in",pad=-120)
        for label in (axs[i_num].get_xticklabels()+axs[i_num].get_yticklabels()):
            label.set_fontsize(22)
        sns.barplot(x=coef_concat.values,y=coef_concat.index,ax=axs[i_num])
        
visualize_coefficient(models)

In [ ]:
test_all_df_dates

In [ ]:
product_ratio_2017 = product_ratio_df.loc[product_ratio_df['date'].dt.year == 2017].copy()
product_ratio_2017['mm-dd'] = product_ratio_2017['date'].dt.strftime('%m-%d')
product_ratio_2017 = product_ratio_2017.drop(columns='date')
product_ratio_2017 = product_ratio_2017.reset_index()

In [ ]:
product_ratio_2018 = product_ratio_df.loc[product_ratio_df['date'].dt.year == 2018].copy()
product_ratio_2018['mm-dd'] = product_ratio_2018['date'].dt.strftime('%m-%d')
product_ratio_2018 = product_ratio_2018.drop(columns='date')
product_ratio_2018 = product_ratio_2018.reset_index()

In [ ]:
product_ratio_2019 = product_ratio_df.loc[product_ratio_df['date'].dt.year == 2019].copy()
product_ratio_2019['mm-dd'] = product_ratio_2019['date'].dt.strftime('%m-%d')
product_ratio_2019 = product_ratio_2019.drop(columns='date')

In [ ]:
product_ratio_2019 = product_ratio_2019.reset_index()

In [ ]:
product_ratio_2019['mean_ratios'] = (product_ratio_2017['ratios']+product_ratio_2018['ratios']+product_ratio_2019['ratios'])/3
product_ratio_2019

In [ ]:
test_product_ratio_df = test_df.copy()
test_product_ratio_df['mm-dd'] = test_product_ratio_df['date'].dt.strftime('%m-%d')

test_product_ratio_df = pd.merge(test_product_ratio_df,product_ratio_2019, how="left", on = ["mm-dd","product"])
test_product_ratio_df.head()

In [ ]:
temp_df = pd.concat([product_ratio_df,test_product_ratio_df]).reset_index(drop=True)
f,ax = plt.subplots(figsize=(20,10))
sns.lineplot(data=temp_df, x="date", y="ratios", hue="product");

In [ ]:
test_sub_df = pd.merge(test_df, test_all_df_dates, how="left")
test_sub_df["ratios"] = test_product_ratio_df["ratios"]
test_sub_df["mean_ratios"] = test_product_ratio_df["mean_ratios"]
test_sub_df.head()

In [ ]:
original_train_df.head()

In [ ]:
store_weights = original_train_df.groupby('store')['num_sold'].sum()/original_train_df['num_sold'].sum()
store_weights

In [ ]:
country_weights = original_train_df.groupby('country')['num_sold'].sum()/original_train_df['num_sold'].sum()
country_weights

In [ ]:
def disaggregate_forecast(df) :
    new_df = df.copy()
    
    store_weights = original_train_df.groupby('store')['num_sold'].sum()/original_train_df['num_sold'].sum()
    print(store_weights)
    country_weights = pd.Series(index = test_sub_df["country"].unique(),data = 1/6)
    print(country_weights)
    for country in country_weights.index:
        new_df.loc[(new_df["country"] == country), "num_sold"] = new_df.loc[(new_df["country"] == country), "num_sold"] *  country_weights[country]
    print(new_df)
    for store in store_weights.index:
        new_df.loc[new_df["store"] == store, "num_sold"] = new_df.loc[new_df["store"] == store, "num_sold"] * store_weights[store]
        
    #new_df["num_sold"] = new_df["num_sold"] * new_df["ratios"]
    new_df["num_sold"] = new_df["num_sold"] * new_df["mean_ratios"]
    new_df["num_sold"] = new_df["num_sold"].round()
    new_df = new_df.drop(columns=["ratios"])    
    
    return new_df

In [ ]:
final_df = disaggregate_forecast(test_sub_df)
final_df

In [ ]:
submission = pd.read_csv("../input/tabular-playground-series-sep-2022/sample_submission.csv")
submission["num_sold"] = final_df["num_sold"]

In [ ]:
submission.to_csv('submission.csv', index = False)

In [ ]:
submission